## XGBoost v. NN baseline

**Config**: n_train at 100k, n_test at 10k, $\sigma \in [0.1, 0.5],$ with $r, T$ default values. All hyperparameters at default.

This experiment determines the performance (RMSE, MAE) of both a neural network and a standard XGBoost model across log moneyness regions. The standard regions used are as follows:
- ITM: $lm = \log \left ( \frac{S}{K} \right ) \in [0.5, 2.0)$
- ATM: $lm \in [-0.2, 0.2]$
- OTM: $lm \in [-2, -0.5]$

In [ ]:
import sys, os, torch

sys.path.append("..")
torch.set_num_threads(1)
torch.set_num_interop_threads(1)

from data.bs_dataset_generator import DataConfig, generate_bundle

# data generation code
base_cfg = DataConfig(
    n_train=100_000,
    n_test=10_000
    )

regions = {
    "ATM, low T": ((-0.2, 0.2), (0.01, 0.25)),
    "ITM, low T": ((0.5, 2.0), (0.01, 0.25)),
    "OTM, low T": ((-2, -0.5), (0.01, 0.25)),
    "ATM, mid T": ((-0.2, 0.2), (0.25, 1.0)),
    "ITM, mid T": ((0.5, 2.0), (0.25, 1.0)),
    "OTM, mid T": ((-2, -0.5), (0.25, 1.0)),
    "ATM, high T": ((-0.2, 0.2), (1.0, 2.0)),
    "ITM, high T": ((0.5, 2.0), (1.0, 2.0)),
    "OTM, high T": ((-2, -0.5), (1.0, 2.0)),
}

bundles = {
}

for region, lm_range in regions.items():
    cfg = base_cfg.replace(log_moneyness_range=lm_range[0])
    cfg = cfg.replace(t_range=lm_range[1])
    bundles[region] = generate_bundle(cfg)

### Hyperparameter tuning

Small grid search on a standard-distribution bundle (`log_m ∈ [-2, 2]`). We pick the config with lowest validation MAE for each model, then reuse those winners across all regimes below.

- XGBoost grid: `max_depth ∈ {5, 7}`, `learning_rate ∈ {0.03, 0.1}`, `n_estimators ∈ {500, 1000}` — 8 configs
- NN grid: `lr ∈ {1e-3, 3e-4}`, `loss_alpha ∈ {0.2, 0.5, 0.8}` — 6 configs

We start with tuning the NN, then the XGB.

In [ ]:
import itertools
import numpy as np
import pandas as pd

from data.bs_dataset_generator import generate_bundle
from training.bs.train_xgboost import train as xgb_train, predict as xgb_predict
from training.bs.train_nn import train as nn_train, predict as nn_predict, set_seed, get_device
from src.eval.validate import compute_metrics


def val_mae(y_true, y_pred):
    return float(np.mean(np.abs(y_pred - y_true)))


tuning_cfg = base_cfg  # base_cfg has log_moneyness_range default = (-2, 2)
tuning_bundle = generate_bundle(tuning_cfg)
Xt, yt = tuning_bundle.train.X, tuning_bundle.train.y_call
Xv, yv = tuning_bundle.val.X,   tuning_bundle.val.y_call

device = get_device(use_cpu=True)
nn_grid = list(itertools.product([1e-3, 3e-4], [0.2, 0.5, 0.8]))
nn_trials = []
for lr, loss_alpha in nn_grid:
    print(f"Training NN: lr={lr}, loss_alpha={loss_alpha}", flush=True)

    set_seed(42)
    params = {"lr": lr, "loss_alpha": loss_alpha}
    model, np_ = nn_train(Xt, yt, Xv, yv, device=device, patience=10, epochs=200, **params)
    y_pred = nn_predict(model, Xv, np_, device="cpu")

    mae = val_mae(yv, y_pred)
    print(f"Done: val_mae={mae}", flush=True)

    nn_trials.append({**params, "val_mae": val_mae(yv, y_pred)})


nn_trials_df = pd.DataFrame(nn_trials).sort_values("val_mae").reset_index(drop=True)
best_nn = {k: v for k, v in nn_trials_df.iloc[0].items() if k != "val_mae"}
print("\nNN trials:")
print(nn_trials_df)
print("Best NN:", best_nn)

Training NN: lr=0.001, loss_alpha=0.2
Epoch 0, Training Loss: 0.81281275, Eval Loss: 0.67444867
Epoch 10, Training Loss: 0.20394008, Eval Loss: 0.19256605
Epoch 20, Training Loss: 0.09393024, Eval Loss: 0.08344009
Epoch 30, Training Loss: 0.03810068, Eval Loss: 0.03386337
Epoch 40, Training Loss: 0.02244359, Eval Loss: 0.02221300
Epoch 50, Training Loss: 0.02008761, Eval Loss: 0.01978509
Epoch 60, Training Loss: 0.01613818, Eval Loss: 0.01556409
Epoch 70, Training Loss: 0.01374032, Eval Loss: 0.01347574
Epoch 80, Training Loss: 0.01195281, Eval Loss: 0.01163852
Epoch 90, Training Loss: 0.01039442, Eval Loss: 0.01014276
Epoch 100, Training Loss: 0.00908455, Eval Loss: 0.00888657
Epoch 110, Training Loss: 0.00799675, Eval Loss: 0.00781123
Epoch 120, Training Loss: 0.00702367, Eval Loss: 0.00684017
Epoch 130, Training Loss: 0.00608508, Eval Loss: 0.00590230
Epoch 140, Training Loss: 0.00516003, Eval Loss: 0.00497978
Epoch 150, Training Loss: 0.00431254, Eval Loss: 0.00413935
Epoch 160, Tr

In [3]:
xgb_grid = list(itertools.product([5, 7], [0.03, 0.1], [500, 1000]))
xgb_trials = []
for max_depth, lr, n_estimators in xgb_grid:
    params = {"max_depth": max_depth, "learning_rate": lr, "n_estimators": n_estimators}
    model, np_ = xgb_train(Xt, yt, Xv, yv, **params)
    y_pred = xgb_predict(model, Xv, np_)
    xgb_trials.append({**params, "val_mae": val_mae(yv, y_pred)})


xgb_trials_df = pd.DataFrame(xgb_trials).sort_values("val_mae").reset_index(drop=True)
best_xgb = {k: v for k, v in xgb_trials_df.iloc[0].items() if k != "val_mae"}
best_xgb["max_depth"] = int(best_xgb["max_depth"])
best_xgb["n_estimators"] = int(best_xgb["n_estimators"])
print("XGBoost trials:")
print(xgb_trials_df)
print("Best XGBoost:", best_xgb)

[0]	validation_0-rmse:0.96932
[50]	validation_0-rmse:0.32067
[100]	validation_0-rmse:0.14900
[150]	validation_0-rmse:0.10523
[200]	validation_0-rmse:0.08974
[250]	validation_0-rmse:0.08426
[300]	validation_0-rmse:0.07918
[350]	validation_0-rmse:0.07647
[400]	validation_0-rmse:0.07349
[450]	validation_0-rmse:0.07019
[499]	validation_0-rmse:0.06768
[0]	validation_0-rmse:0.96932
[50]	validation_0-rmse:0.32067
[100]	validation_0-rmse:0.14900
[150]	validation_0-rmse:0.10523
[200]	validation_0-rmse:0.08974
[250]	validation_0-rmse:0.08426
[300]	validation_0-rmse:0.07918
[350]	validation_0-rmse:0.07647
[400]	validation_0-rmse:0.07349
[450]	validation_0-rmse:0.07019
[500]	validation_0-rmse:0.06768
[550]	validation_0-rmse:0.06667
[600]	validation_0-rmse:0.06523
[650]	validation_0-rmse:0.06455
[700]	validation_0-rmse:0.06353
[750]	validation_0-rmse:0.06234
[800]	validation_0-rmse:0.06168
[850]	validation_0-rmse:0.06088
[900]	validation_0-rmse:0.05980
[950]	validation_0-rmse:0.05890
[999]	validati

## Training and Eval

We train both models on the ranges mentioned earlier. Subsequently, we compare the model's prediction metrics across test data seeded from the same ranges as the training data:
- MAE
- RMSE
- Mean and median relative error
- Maximum error

Both models utilize similar feature engineering, opting for log-moneyness instead of standard $S$ and $K$ values. This is because option price $C$ increases roughly monotonically with $\log \left ( \frac{S}{K} \right ).$ For true predictions, we can mathematically compute log moneyness manually.
Furthermore, the NN uses a hybridized loss function, linearly interpolating standard MSE with a $y-$weighted MSE. This is done to minimize both relative error at OTM, and absolute error at ITM.

In [ ]:
import pandas as pd
from training.bs.train_nn import train, set_seed, get_device, predict
from src.eval.validate import preprocess_test_data, compute_metrics

set_seed(42) 
device = get_device(use_cpu=True)

results = []

for region in bundles:
    # Train only on calls, for now
    bundle = bundles[region]
    X_train, y_train = bundle.train.X, bundle.train.y_call
    X_val, y_val = bundle.val.X, bundle.val.y_call
    
    nn_model, norm_params = train(
        X_train, y_train, X_val, y_val,
        device=device,
        patience=25,
        epochs=400,
        **best_nn,
    )

    X_test, y_test = bundle.test.X, bundle.test.y_call
    X_test, _ = preprocess_test_data(X_test)
    y_pred = predict(nn_model, X_test, norm_params, device="cpu")

    metrics = compute_metrics(y_test, y_pred, min_value_threshold=0.01)
    results.append({
        "model" : "NN",
        "regime" : region,
        **{k: v for k, v in metrics.items() if k not in ["errors", "abs_errors", "relative_errors"]}
    })

Epoch 0, Training Loss: 0.48901206, Eval Loss: 0.39717954
Epoch 10, Training Loss: 0.16313906, Eval Loss: 0.14088154
Epoch 20, Training Loss: 0.09742263, Eval Loss: 0.08853713
Epoch 30, Training Loss: 0.04889607, Eval Loss: 0.04366808
Epoch 40, Training Loss: 0.02113146, Eval Loss: 0.01927238
Epoch 50, Training Loss: 0.01403094, Eval Loss: 0.01358098
Epoch 60, Training Loss: 0.01032711, Eval Loss: 0.00992335
Epoch 70, Training Loss: 0.00819530, Eval Loss: 0.00781864
Epoch 80, Training Loss: 0.00702522, Eval Loss: 0.00662562
Epoch 90, Training Loss: 0.00603504, Eval Loss: 0.00565253
Epoch 100, Training Loss: 0.00529786, Eval Loss: 0.00494000
Epoch 110, Training Loss: 0.00474008, Eval Loss: 0.00440631
Epoch 120, Training Loss: 0.00429835, Eval Loss: 0.00397851
Epoch 130, Training Loss: 0.00393405, Eval Loss: 0.00362531
Epoch 140, Training Loss: 0.00363700, Eval Loss: 0.00334294
Epoch 150, Training Loss: 0.00338072, Eval Loss: 0.00309852
Epoch 160, Training Loss: 0.00314996, Eval Loss: 0.

Now to train the XGBoost model:

In [ ]:
import pandas as pd
from training.bs.train_xgboost import train as xgb_train, predict as xgb_predict
from src.eval.validate import preprocess_test_data, compute_metrics

xgb_results = []

for region in bundles:
    bundle = bundles[region]
    X_train, y_train = bundle.train.X, bundle.train.y_call
    X_val, y_val = bundle.val.X, bundle.val.y_call

    xgb_model, norm_params = xgb_train(
        X_train, y_train, 
        X_val, y_val, 
        **best_xgb
    )

    X_test_raw, y_test = bundle.test.X, bundle.test.y_call
    X_test, _ = preprocess_test_data(X_test_raw)
    y_pred = xgb_predict(xgb_model, X_test, norm_params)

    metrics = compute_metrics(y_test, y_pred, min_value_threshold=0.01)
    xgb_results.append({
        "model": "XGBoost",
        "regime": region,
        **{k: v for k, v in metrics.items() if k not in ["errors", "abs_errors", "relative_errors"]}
    })

[0]	validation_0-rmse:0.90643
[50]	validation_0-rmse:0.07360
[100]	validation_0-rmse:0.04540
[150]	validation_0-rmse:0.03833
[200]	validation_0-rmse:0.03354
[250]	validation_0-rmse:0.03129
[300]	validation_0-rmse:0.02933
[350]	validation_0-rmse:0.02771
[400]	validation_0-rmse:0.02616
[450]	validation_0-rmse:0.02507
[500]	validation_0-rmse:0.02437
[550]	validation_0-rmse:0.02387
[600]	validation_0-rmse:0.02333
[650]	validation_0-rmse:0.02305
[700]	validation_0-rmse:0.02271
[750]	validation_0-rmse:0.02245
[800]	validation_0-rmse:0.02220
[850]	validation_0-rmse:0.02203
[900]	validation_0-rmse:0.02158
[950]	validation_0-rmse:0.02135
[999]	validation_0-rmse:0.02121
[0]	validation_0-rmse:0.89712
[50]	validation_0-rmse:0.01450
[100]	validation_0-rmse:0.00484
[150]	validation_0-rmse:0.00466
[200]	validation_0-rmse:0.00458
[250]	validation_0-rmse:0.00453
[300]	validation_0-rmse:0.00451
[350]	validation_0-rmse:0.00450
[400]	validation_0-rmse:0.00448
[450]	validation_0-rmse:0.00447
[500]	validati

Finally, let's compare both!

In [6]:
results_df = pd.DataFrame(results)
xgb_results_df = pd.DataFrame(xgb_results)
X
print(pd.concat([results_df, xgb_results_df], ignore_index=True))

      model       regime           mae          rmse  mean_relative_error  \
0        NN   ATM, low T  1.707882e-03  2.755723e-03             0.034854   
1        NN   ITM, low T  3.492645e-02  4.550886e-02             0.016590   
2        NN   OTM, low T  3.033773e-08  5.467981e-07             0.000000   
3        NN   ATM, mid T  1.135964e-03  1.571385e-03             0.013594   
4        NN   ITM, mid T  3.849890e-02  4.774003e-02             0.018055   
5        NN   OTM, mid T  4.453036e-05  4.111276e-04             0.195069   
6        NN  ATM, high T  1.957552e-03  2.541637e-03             0.017151   
7        NN  ITM, high T  3.977790e-02  5.344782e-02             0.017011   
8        NN  OTM, high T  1.911656e-04  7.225646e-04             0.067859   
9   XGBoost   ATM, low T  1.051041e-03  2.088415e-03             0.019593   
10  XGBoost   ITM, low T  5.977456e-03  7.758421e-03             0.002369   
11  XGBoost   OTM, low T  1.218863e-07  2.651559e-06             0.000000   


The XGBoost model significantly outperforms the neural network, most significantly in the ITM regime. Across all expiry times, we observe an average MAE and RMSE of 0.038, 0.049 respectively, which is roughly 6.3x the average MAE/RMSE of our XRBoost model. Furthermore, in the ATM and OTM regimes, smaller, yet consistent gains (of 1.7x and 3x, respectively) are observed.

This discrepancy is most likely structural, rather than incidental. For instance, in the ITM regime, we have $S >> K \Rightarrow D_1, D_2 \rightarrow \infty$. Thus, we approximate: $N(D_1), N(D_2) \approx 1,$ which means that
$$C \approx S - K e ^ {-rT}.$$ 
In log normalized form, this becomes
$$ \log{\dfrac{C}{K}} \approx \log{ \left ( \dfrac{S}{K} - e^{-rT} \right )}.$$ 

Although each model is trained independently within each regime (thus eliminating cross-regime generalization effects), the performance gap persists. As $S$ grows very large, $\frac{C}{K}$ becomes roughly affine in structure. However, as our neural network is trained in log-space but **evaluated in affine space**, errors become magnitude-dependent, increasing as $\frac{S}{K}$ does.

Furthermore, despite the issue persisting within the XGBoost model, the piecewise nature of its implementation means that each leaf within the model can be arbitrarily accurate, thus allowing them to reduce the structure by region via piecewise partitioning, in contrast to the NN, which is forced to learn a smooth, but global, approximation. Ultimately, this leads to significantly lower error observed across **all** regimes.

Finally, we can observe a noticeable expiry-time dependence in ITM for the neural network: as $T$ increases, both MAE and RMSE are observed to increase monotonically, even while XGBoost remains nearly constant error across all $T$ values. This is possibly due to the nature of the model's training space. As $T$ increases, our approximation for $C$ yields larger values, as $e^{-rT}$ decreases. Thus, small errors in log space may transfer into larger absolute errors as $\frac{C}{K}$ increases, further reinforcing the structural nature of the discrepancy.

One limitation of this study lies in the the mismatch between the training objective and evaluation metric, as outlined above. Further work could address such limitations by aligning training and evaluation objectives (i.e. evaluation in log-space), providing a more comprehensive comparison of model capabilities across the options-pricing function.

## Model Comparison Conclusion

Note that many of these results were also taken from ablations, as done in other notebooks. Results are aggregated across the full test distribution (log-moneyness ∈ [-2, 2], T ∈ [0.01, 2]).

| Model | MAE | RMSE | Max Error | Mean Rel. | Median Rel. |
|-------|-----|------|-----------|-----------|-------------|
| NN baseline (log-m, standardized, hybrid loss) | 0.0478 | 0.1075 | 1.897 | 8.39% | 4.82% |
| NN no feature eng. (raw S, K) | 0.1637 | 0.3569 | 3.602 | 20.64% | 14.88% |
| NN reduced width (4-64-64-64-1) | 0.0756 | 0.1959 | 3.911 | 14.91% | 6.77% |
| NN reduced depth (4-128-128-1) | 0.0869 | 0.1712 | 1.588 | 16.97% | 9.15% |
| NN no standardization | 0.0165 | 0.0261 | 0.224 | 8.69% | 1.53% |
| NN no hybrid loss | 0.0607 | 0.1497 | 2.310 | 8.13% | 4.94% |
| XGB baseline (log-m, standardized) | 0.0308 | 0.0754 | 1.075 | 4.24% | 2.45% |
| XGB no feature eng. (raw S, K) | 0.0973 | 0.2125 | 3.143 | 21.02% | 9.21% |
| **XGB no standardization** | **0.0077** | **0.0124** | **0.068** | **2.47%** | **0.74%** |

### Takeaways

- XGB without standardization is the clear Stage 1 winner.** MAE of 0.0077 and max error of 0.068, which is essentially a perfect fit on this problem. Furthermore, performance is roughly 4x better than the next-best model (NN no standardization).
- Standardization decreases metrics over **both models.** The log-transform + z-score warps the target into a space where the loss landscape is harder to optimize, without the benefit it would provide on noisy or skewed distributions.
- Feature engineering is also significant in improving error on both models Without it, both NN and XGB degrade 3-5x. The model must otherwise learn that only S/K matters and reconstruct the log structure internally.
- Architecture on the NN affects magnitude but not shape. Smaller architectures produce the same error pattern at ~2x scale, suggesting the residual error is structural to the MLP, not a capacity issue.
- Hybrid loss acts as expected by improves tail accuracy at the expense of aggregate metrics. Removing it makes headline MAE slightly worse and tails significantly worse.

However, on real data with noise, skew, and microstructure, standardization will likely regain value. The XGB dominance is also expected to diminish when the target function becomes stochastic.